# Inez CO₂ Storage Capacity Assessment

This notebook simulates the Haldager Sand, Gassum and Skagerrak reservoirs separately, then adds their capacity samples trial by trial to obtain the combined Inez distribution. Inputs come from GEUS Report 2022/29, Tables 8.1.5.1–8.1.5.3.

In [ ]:
%pip install -q --upgrade --force-reinstall --no-cache-dir --no-deps "git+https://github.com/AnaSoles/ggg-co2-storage-eval.git"
%pip install -q matplotlib pandas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from storageeval import Distribution, SimulationResult, StorageSite, simulate
plt.style.use("seaborn-v0_8-whitegrid")

## Input values and the 7% efficiency mode

Fractions are decimals: `0.07` means 7%. In `PERT(minimum, mode, maximum)`, the mode is the most likely input value. It is sampled **before** calculating capacity; it is not P50 and is not applied after P90/P50/P10. The Inez Gassum reservoir-specific table gives a 7% mode. This value reproduces the published result more closely than the 10% mentioned in the general text.

In [ ]:
inputs = {
    "Haldager Sand": {
        "GRV (km³)": (0.114, 0.417, 1.676), "N/G": (0.200, 0.317, 0.500),
        "Porosity": (0.200, 0.255, 0.300), "CO₂ density (kg/m³)": (609.6, 641.7, 705.9),
        "Storage efficiency": (0.050, 0.100, 0.150)},
    "Gassum Formation": {
        "GRV (km³)": (11.059, 28.076, 49.666), "N/G": (0.4696, 0.5870, 0.7044),
        "Porosity": (0.1624, 0.2030, 0.2436), "CO₂ density (kg/m³)": (611.8, 644.0, 708.4),
        "Storage efficiency": (0.050, 0.070, 0.150)},
    "Skagerrak Formation": {
        "GRV (km³)": (3.7105, 8.781, 12.915), "N/G": (0.3056, 0.3820, 0.4584),
        "Porosity": (0.1624, 0.2030, 0.2436), "CO₂ density (kg/m³)": (607.2, 639.2, 703.1),
        "Storage efficiency": (0.050, 0.100, 0.150)},
}
rows = [[reservoir, parameter, "PERT", *values] for reservoir, parameters in inputs.items() for parameter, values in parameters.items()]
input_table = pd.DataFrame(rows, columns=["Reservoir", "Parameter", "Distribution", "Minimum", "Mode", "Maximum"])
input_table

In [ ]:
def make_site(name, values):
    return StorageSite(
        name=f"Inez – {name}",
        grv=Distribution.pert(*values["GRV (km³)"]),
        net_to_gross=Distribution.pert(*values["N/G"]),
        porosity=Distribution.pert(*values["Porosity"]),
        co2_density=Distribution.pert(*values["CO₂ density (kg/m³)"]),
        storage_efficiency=Distribution.pert(*values["Storage efficiency"]),
    )

iterations = 100_000
results = {name: simulate(make_site(name, values), iterations, seed=42+i) for i, (name, values) in enumerate(inputs.items())}
combined_capacity = np.sum([result.capacity_mt for result in results.values()], axis=0)
combined = SimulationResult("Inez – combined reservoirs", combined_capacity, {})

## Comparison with GEUS

P90/P50/P10 are calculated from each final capacity distribution. For combined Inez, we add the three reservoirs in every trial and then calculate percentiles; we do not add reservoir percentiles.

In [ ]:
published = {
    "Haldager Sand": (1.2, 2.8, 5.5, 3.1),
    "Gassum Formation": (103.8, 168.1, 263.7, 177.6),
    "Skagerrak Formation": (27.4, 42.1, 60.7, 43.2),
    "Combined Inez": (148.6, 216.2, 310.2, 224.8),
}
all_results = {**results, "Combined Inez": combined}
comparison_rows = []
for name, result in all_results.items():
    s = result.summary()
    simulated = (s["p90_mt"], s["p50_mt"], s["p10_mt"], s["mean_mt"])
    comparison_rows.append([name, *simulated, *published[name]])
pd.DataFrame(comparison_rows, columns=["Reservoir", "P90 simulated", "P50 simulated", "P10 simulated", "Mean simulated", "P90 GEUS", "P50 GEUS", "P10 GEUS", "Mean GEUS"]).round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(combined.capacity_mt, bins=60, density=True, color="#2a6fbb", alpha=0.82)
summary = combined.summary()
for key, color in [("p90_mt", "#2ca02c"), ("p50_mt", "#ffbf00"), ("p10_mt", "#d62728")]:
    axes[0].axvline(summary[key], color=color, label=f"{key[:3].upper()}: {summary[key]:.1f} Mt")
axes[0].set(title="Combined Inez capacity distribution", xlabel="Capacity (Mt CO₂)", ylabel="Probability density")
axes[0].legend()
capacity = np.sort(combined.capacity_mt)
exceedance = 1 - np.arange(1, capacity.size + 1) / (capacity.size + 1)
axes[1].plot(capacity, exceedance * 100, color="#2a6fbb")
axes[1].set(title="Combined Inez exceedance curve", xlabel="Capacity (Mt CO₂)", ylabel="Exceedance probability (%)")
fig.tight_layout(); plt.show()

## Source and limitation

Source: [GEUS Report 2022/29](https://data.geus.dk/pure-pdf/GEUS-R_2022-29_web.pdf), input Tables 8.1.5.1–8.1.5.3 (report page 44) and results Tables 8.2.1–8.2.4 (page 46). This is static volumetric screening capacity, not dynamically constrained injectable capacity.